# NB14 — Benchmark Parity, Failure Decomposition, and Hybrid Blueprint

This notebook is the **decision notebook** after NB10–NB13.

Its purpose is to do four things without overclaiming:

1. Put **NB11 residual**, **NB12/NB13 OT**, and **chemCPA (if available)** into one standardized comparison table.
2. Perform the **most important failure decomposition** that current saved artifacts allow.
3. Give an **honest dataset adequacy readout**: whether the current dataset is suitable for a rigorous scaffold/OOD benchmark study.
4. Produce a **hybrid-method blueprint** for the next benchmark-beating attempt.

This notebook is **not** a new method notebook.  
It is a **truth-finding notebook**.

## What this notebook will and will not claim

This notebook can support claims such as:

- the dataset is or is not adequate for a serious scaffold/OOD benchmark study
- the current internal baseline is or is not stable
- OT is or is not providing meaningful gains over internal references
- gains are concentrated in particular cell types / supergroups / dose regions / drug families

This notebook will **not** claim:

- “new SOTA” without **same-benchmark chemCPA parity**
- “universal superiority” from one dataset
- “strong biological validation” if the required pathway artifacts are missing or broken

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy rdkit gseapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

In [3]:
import os
import json
import pickle
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

In [4]:
@dataclass
class CFG:
    # core data
    data_path: str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"

    # prior notebook dirs
    nb10_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold"
    nb10_art_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts"

    nb11_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb11_scaffold"
    nb12_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot"
    nb12_art_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot/artifacts"
    nb13_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb13"

    # optional chemCPA parity artifacts
    chemcpa_res_dir: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold"
    chemcpa_metrics_csv: str = "/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_scaffold_metrics.csv"

    # new notebook output
    results_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb14_benchmark_parity"
    art_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb14_benchmark_parity/artifacts"

    # schema
    split_col: str = "split_scaffold"
    condition_col: str = "condition"
    cell_col: str = "cell_type"
    dose_col: str = "dose"
    fallback_dose_col: str = "dose_val"
    control_label: str = "control"

    # optional family / moa-like columns to try
    family_candidates: tuple = (
        "pathway_level_1", "pathway_level_2", "pathway",
        "target", "product_name", "moa", "mechanism_of_action"
    )

    # hybrid blueprint thresholds
    ot_gain_threshold_overall: float = 0.01
    ot_gain_threshold_group: float = 0.03

cfg = CFG()
Path(cfg.results_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.art_dir).mkdir(parents=True, exist_ok=True)

print("NB14 config:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")

NB14 config:
  data_path: /content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad
  nb10_res_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold
  nb10_art_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts
  nb11_res_dir: /content/drive/MyDrive/ChemDFM/results_nb11_scaffold
  nb12_res_dir: /content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot
  nb12_art_dir: /content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot/artifacts
  nb13_res_dir: /content/drive/MyDrive/ChemDFM/results_nb13
  chemcpa_res_dir: /content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold
  chemcpa_metrics_csv: /content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_scaffold_metrics.csv
  results_dir: /content/drive/MyDrive/ChemDFM/results_nb14_benchmark_parity
  art_dir: /content/drive/MyDrive/ChemDFM/results_nb14_benchmark_parity/artifacts
  split_col: split_scaffold
  condition_col: condition
  cell_col: cell_type
  dose_col: dose
  fallback_dose_col: dose_val
  control_la

## Load data and artifact audit

This cell performs a **strict audit** of the artifacts saved by NB10–NB13.

It is acceptable if chemCPA artifacts are missing at this stage.  
If they are missing, the notebook will still run, but it will explicitly say that **benchmark parity is incomplete**.

In [8]:
def read_json(path):
    if path is None or not os.path.exists(path):
        return None
    with open(path, "r") as f:
        return json.load(f)


def read_csv(path):
    if path is None or not os.path.exists(path):
        return None
    return pd.read_csv(path)


def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def first_existing_path(paths):
    for p in paths:
        if p is not None and os.path.exists(p):
            return p
    return None


# ── load adata ───────────────────────────────────────────────────────────
adata = ad.read_h5ad(cfg.data_path)
if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose_col in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]

# ── locate NB10 artifacts robustly ───────────────────────────────────────
nb10_gate_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "scaffold_split_gate.json"),
    os.path.join(cfg.nb10_art_dir, "scaffold_split_gate.json"),
])

scaffold_map_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "scaffold_split_map.csv"),
    os.path.join(cfg.nb10_art_dir, "scaffold_split_map.csv"),
])

obs_nb10_path = first_existing_path([
    os.path.join(cfg.nb10_res_dir, "obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_art_dir, "obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_res_dir, "adata_obs_with_scaffold_split.csv"),
    os.path.join(cfg.nb10_art_dir, "adata_obs_with_scaffold_split.csv"),
])

nb10_gate = read_json(nb10_gate_path)
scaffold_map = read_csv(scaffold_map_path)
obs_nb10 = read_csv(obs_nb10_path)

print("NB10 gate path:", nb10_gate_path)
print("Scaffold map path:", scaffold_map_path)
print("NB10 obs path:", obs_nb10_path)

# ── recover split_scaffold if missing in original h5ad ──────────────────
if cfg.split_col not in adata.obs.columns:
    recovered = False

    # Option A: recover from full obs table saved by NB10
    if obs_nb10 is not None:
        candidate_index_cols = ["obs_index", "index", "Unnamed: 0"]
        found_index_col = None
        for c in candidate_index_cols:
            if c in obs_nb10.columns:
                found_index_col = c
                break

        if found_index_col is not None and cfg.split_col in obs_nb10.columns:
            split_map = obs_nb10[[found_index_col, cfg.split_col]].copy()
            split_map = split_map.rename(columns={found_index_col: "_obs_index_nb14"})
            split_map["_obs_index_nb14"] = split_map["_obs_index_nb14"].astype(str)

            adata.obs["_obs_index_nb14"] = adata.obs.index.astype(str)
            adata.obs = adata.obs.merge(split_map, on="_obs_index_nb14", how="left")
            adata.obs.index = adata.obs["_obs_index_nb14"].astype(str)

            if not adata.obs[cfg.split_col].isna().any():
                recovered = True
                print(f"Recovered '{cfg.split_col}' from NB10 obs table.")

    # Option B: recover from drug-level scaffold map
    if not recovered and scaffold_map is not None:
        if cfg.split_col not in scaffold_map.columns:
            raise RuntimeError(
                f"scaffold_map found at {scaffold_map_path}, but it does not contain '{cfg.split_col}'. "
                f"Available columns: {list(scaffold_map.columns)}"
            )

        drug_col = choose_first_existing(
            list(scaffold_map.columns),
            [cfg.condition_col, "drug", "drug_name", "condition", "compound", "perturbation"]
        )
        if drug_col is None:
            raise RuntimeError(
                f"scaffold_map found at {scaffold_map_path}, but no drug/condition column was detected. "
                f"Available columns: {list(scaffold_map.columns)}"
            )

        cond_series = adata.obs[cfg.condition_col].astype(str)
        adata.obs[cfg.split_col] = np.where(
            cond_series == cfg.control_label,
            cfg.control_label,
            cond_series.map(
                scaffold_map[[drug_col, cfg.split_col]]
                .drop_duplicates()
                .set_index(drug_col)[cfg.split_col]
            )
        )

        missing_mask = adata.obs[cfg.split_col].isna() & (cond_series != cfg.control_label)
        if missing_mask.any():
            missing_drugs = sorted(cond_series[missing_mask].unique().tolist())[:20]
            raise RuntimeError(
                f"Recovered split labels from scaffold_map, but some perturbed drugs are still missing. "
                f"Example missing drugs: {missing_drugs}"
            )

        recovered = True
        print(
            f"Recovered '{cfg.split_col}' from scaffold_map using drug column '{drug_col}'."
        )

    if not recovered:
        raise RuntimeError(
            f"Missing '{cfg.split_col}' in adata.obs, and NB10 recovery artifacts were not usable. "
            f"Looked for:\n"
            f"  gate: {nb10_gate_path}\n"
            f"  scaffold_map: {scaffold_map_path}\n"
            f"  obs table: {obs_nb10_path}\n"
            "This usually means cfg.nb10_res_dir / cfg.nb10_art_dir is pointing to the wrong folder."
        )

# ── schema check after recovery ──────────────────────────────────────────
required_obs = [cfg.split_col, cfg.condition_col, cfg.cell_col, cfg.dose_col]
missing = [c for c in required_obs if c not in adata.obs.columns]
if missing:
    raise RuntimeError(f"Missing required obs columns even after NB10 recovery: {missing}")

# optional family/moa column
family_col = choose_first_existing(list(adata.obs.columns), cfg.family_candidates)

# ── load NB11 artifacts ──────────────────────────────────────────────────
nb11_summary = read_json(os.path.join(cfg.nb11_res_dir, "nb11_run_summary.json"))
nb11_overall = read_csv(os.path.join(cfg.nb11_res_dir, "final_overall_residual_only.csv"))
nb11_per_cell = read_csv(os.path.join(cfg.nb11_res_dir, "final_per_cell_residual_only.csv"))

# ── load NB12 artifacts ──────────────────────────────────────────────────
nb12_summary = read_json(os.path.join(cfg.nb12_res_dir, "run_summary_nb12.json"))
nb12_overall = read_csv(os.path.join(cfg.nb12_res_dir, "eval_overall_nb12.csv"))
nb12_per_cell = read_csv(os.path.join(cfg.nb12_res_dir, "eval_per_cell_nb12.csv"))
nb12_per_sg = read_csv(os.path.join(cfg.nb12_res_dir, "eval_per_sg_nb12.csv"))
nb12_routing = read_csv(os.path.join(cfg.nb12_art_dir, "ood_test_drug_routing.csv"))
nb12_train_sg = read_csv(os.path.join(cfg.nb12_art_dir, "train_drug_supergroup_assignments.csv"))

# ── load NB13 artifacts ──────────────────────────────────────────────────
nb13_ci = read_csv(os.path.join(cfg.nb13_res_dir, "ci_table_nb13.csv"))
nb13_ci_per_cell = read_csv(os.path.join(cfg.nb13_res_dir, "ci_per_cell_nb13.csv"))
nb13_supergroup = read_csv(os.path.join(cfg.nb13_res_dir, "supergroup_ood_analysis_nb13.csv"))
nb13_pathway = read_csv(os.path.join(cfg.nb13_res_dir, "pathway_recovery_nb13.csv"))

# ── optional chemCPA artifacts ───────────────────────────────────────────
chemcpa_metrics = read_csv(cfg.chemcpa_metrics_csv)

artifact_rows = [
    ("NB10 gate", nb10_gate is not None),
    ("NB10 scaffold map", scaffold_map is not None),
    ("NB10 obs table", obs_nb10 is not None),
    ("NB11 run summary", nb11_summary is not None),
    ("NB11 overall metrics", nb11_overall is not None),
    ("NB11 per-cell metrics", nb11_per_cell is not None),
    ("NB12 run summary", nb12_summary is not None),
    ("NB12 overall metrics", nb12_overall is not None),
    ("NB12 per-cell metrics", nb12_per_cell is not None),
    ("NB12 per-supergroup metrics", nb12_per_sg is not None),
    ("NB12 routing table", nb12_routing is not None),
    ("NB12 train supergroups", nb12_train_sg is not None),
    ("NB13 CI table", nb13_ci is not None),
    ("NB13 CI per-cell", nb13_ci_per_cell is not None),
    ("NB13 supergroup analysis", nb13_supergroup is not None),
    ("NB13 pathway table", nb13_pathway is not None),
    ("chemCPA metrics", chemcpa_metrics is not None),
]
artifact_audit = pd.DataFrame(artifact_rows, columns=["artifact", "available"])
display(artifact_audit)

artifact_audit.to_csv(
    os.path.join(cfg.results_dir, "artifact_audit_nb14.csv"),
    index=False
)

NB10 gate path: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/scaffold_split_gate.json
Scaffold map path: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/scaffold_split_map.csv
NB10 obs path: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts/obs_with_scaffold_split.csv
Recovered 'split_scaffold' from NB10 obs table.


,artifact,available
0,NB10 gate,True
1,NB10 scaffold map,True
2,NB10 obs table,True
3,NB11 run summary,True
4,NB11 overall metrics,True
5,NB11 per-cell metrics,True
6,NB12 run summary,True
7,NB12 overall metrics,True
8,NB12 per-cell metrics,True
9,NB12 per-supergroup metrics,True


## Dataset adequacy: is the current dataset okay?

This cell gives an **honest dataset readout**.

A dataset can be “okay” for a serious benchmark study even if it is **not enough** for a universal SOTA claim.

In [9]:
obs = adata.obs.copy()

pert_mask = obs[cfg.condition_col].astype(str) != cfg.control_label
train_mask = obs[cfg.split_col].astype(str) == "train"
test_mask  = obs[cfg.split_col].astype(str) == "test"
ood_mask   = obs[cfg.split_col].astype(str) == "ood"
ctrl_mask  = obs[cfg.split_col].astype(str) == cfg.control_label

dataset_summary = {
    "n_cells_total": int(adata.n_obs),
    "n_genes_total": int(adata.n_vars),
    "n_cell_types": int(obs[cfg.cell_col].nunique()),
    "cell_types": sorted(map(str, obs[cfg.cell_col].unique())),
    "n_perturbed_cells": int(pert_mask.sum()),
    "n_control_cells": int(ctrl_mask.sum()),
    "n_train_cells": int(train_mask.sum()),
    "n_test_cells": int(test_mask.sum()),
    "n_ood_cells": int(ood_mask.sum()),
    "n_unique_conditions_total": int(obs[cfg.condition_col].nunique()),
    "n_unique_perturbed_drugs": int(obs.loc[pert_mask, cfg.condition_col].nunique()),
    "dose_min": float(pd.to_numeric(obs[cfg.dose_col], errors="coerce").min()),
    "dose_max": float(pd.to_numeric(obs[cfg.dose_col], errors="coerce").max()),
    "family_col_used": family_col,
}
if scaffold_map is not None:
    dataset_summary["n_scaffold_families"] = int(scaffold_map["scaffold_key"].nunique()) if "scaffold_key" in scaffold_map.columns else np.nan

dataset_df = pd.DataFrame([dataset_summary]).T.reset_index()
dataset_df.columns = ["field", "value"]
display(dataset_df)

# split x cell type
split_cell_table = (
    obs.groupby([cfg.split_col, cfg.cell_col]).size().reset_index(name="n_cells")
    .pivot(index=cfg.cell_col, columns=cfg.split_col, values="n_cells")
    .fillna(0).astype(int)
)
display(split_cell_table)

# split x dose bin
dose_values = pd.to_numeric(obs[cfg.dose_col], errors="coerce")
obs["_dose_bin_nb14"] = pd.qcut(dose_values.rank(method="first"), q=5, labels=[f"Q{i}" for i in range(1,6)])
dose_split_table = (
    obs.groupby([cfg.split_col, "_dose_bin_nb14"]).size().reset_index(name="n_cells")
    .pivot(index="_dose_bin_nb14", columns=cfg.split_col, values="n_cells")
    .fillna(0).astype(int)
)
display(dose_split_table)

dataset_df.to_csv(os.path.join(cfg.results_dir, "dataset_summary_nb14.csv"), index=False)
split_cell_table.to_csv(os.path.join(cfg.results_dir, "split_by_celltype_nb14.csv"))
dose_split_table.to_csv(os.path.join(cfg.results_dir, "split_by_dosebin_nb14.csv"))

# honest adequacy note
adequacy_lines = []
adequacy_lines.append("Dataset adequacy verdict:")
adequacy_lines.append("- Suitable for a serious scaffold/OOD benchmark study: YES, if artifacts remain leakage-safe.")
adequacy_lines.append("- Suitable for a universal SOTA claim across perturbation biology: NO, not by itself.")
adequacy_lines.append("- Why okay: large cell count, multiple cell types, many drugs, scaffold split already survived.")
adequacy_lines.append("- Why limited: single dataset, only 3 cell types, benchmark-specific rather than universal.")
print("\n".join(adequacy_lines))

,field,value
0,n_cells_total,354640
1,n_genes_total,2000
2,n_cell_types,3
3,cell_types,"[A549, K562, MCF7]"
4,n_perturbed_cells,341636
5,n_control_cells,13004
6,n_train_cells,240527
7,n_test_cells,51243
8,n_ood_cells,49866
9,n_unique_conditions_total,188


split_scaffold,control,ood,test,train
cell_type,,,,
A549,3287,11936,12626,59862
K562,3359,13062,12522,60256
MCF7,6358,24868,26095,120409


split_scaffold,control,ood,test,train
_dose_bin_nb14,,,,
Q1,13004,8401,8523,41000
Q2,0,10021,10608,50299
Q3,0,10233,10708,49987
Q4,0,10324,10639,49965
Q5,0,10887,10765,49276


Dataset adequacy verdict:
- Suitable for a serious scaffold/OOD benchmark study: YES, if artifacts remain leakage-safe.
- Suitable for a universal SOTA claim across perturbation biology: NO, not by itself.
- Why okay: large cell count, multiple cell types, many drugs, scaffold split already survived.
- Why limited: single dataset, only 3 cell types, benchmark-specific rather than universal.


## Standardized benchmark parity table

This section puts all currently available methods into a single comparable table.

Important caveat:
- **NB11 vs NB12/NB13** here is a **cross-notebook metric comparison** on the same split/metric family.
- That is useful and fair enough for strategic decisions.
- But it is **not** a paired per-sample significance test.

In [10]:
def extract_nb11_overall(summary_json, overall_df):
    rows = []
    # preferred: CSV if available
    if overall_df is not None and not overall_df.empty:
        df = overall_df.copy()
        df.columns = [str(c) for c in df.columns]
        return df
    # fallback: parse summary JSON
    if summary_json is None:
        return pd.DataFrame()
    rec = summary_json.get("residual_only", {})
    out = []
    for split in ["test", "ood"]:
        row = {"method": "nb11_residual", "split": split}
        for metric in ["r2_top50", "r2_top20", "pearson_row", "r2_full", "mse"]:
            key = f"{split}_{metric}"
            if key in rec:
                row[metric] = rec[key]
        out.append(row)
    return pd.DataFrame(out)

def standardize_nb11(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    out = []
    # robust to many likely formats
    for _, r in df.iterrows():
        row = {k: r[k] for k in df.columns if k in ["split","r2_top50","r2_top20","pearson_row","r2_full","mse","n_cells"]}
        if "split" not in row:
            label = str(r.get("label",""))
            if "test" in label: row["split"] = "test"
            elif "ood" in label: row["split"] = "ood"
        row["method"] = "nb11_residual"
        row["source"] = "NB11"
        out.append(row)
    return pd.DataFrame(out)

def standardize_nb12(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    out = []
    for _, r in df.iterrows():
        label = str(r["label"])
        if label.startswith("ot_"):
            split = label.replace("ot_", "")
            method = "nb12_ot"
        elif "ctrl_baseline_" in label:
            split = label.replace("ctrl_baseline_", "")
            method = "ctrl_baseline"
        else:
            continue
        out.append({
            "method": method,
            "split": split,
            "r2_top50": r.get("r2_top50", np.nan),
            "r2_top20": r.get("r2_top20", np.nan),
            "pearson_row": r.get("pearson_row", np.nan),
            "r2_full": r.get("r2_full", np.nan),
            "mse": r.get("mse", np.nan),
            "n_cells": r.get("n_cells", np.nan),
            "source": "NB12",
        })
    return pd.DataFrame(out)

def standardize_nb13(ci_df):
    if ci_df is None or ci_df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    piv = (
        ci_df.pivot_table(index=["split","method"], columns="metric", values="mean", aggfunc="first")
        .reset_index()
        .rename(columns={"method":"method_raw"})
    )
    piv["method"] = piv["method_raw"].map({"ot":"nb13_ot_mean", "ctrl":"ctrl_baseline"})
    piv["source"] = "NB13"
    return piv[["method","split","r2_top50","r2_top20","pearson_row","source"]]

def standardize_chemcpa(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["method","split","r2_top50","r2_top20","pearson_row","source"])
    x = df.copy()
    x.columns = [str(c) for c in x.columns]
    required = {"method","split"}
    if not required.issubset(set(x.columns)):
        raise RuntimeError(
            "chemCPA metrics CSV must contain at least columns: method, split. "
            "Recommended columns: r2_top50, r2_top20, pearson_row, r2_full, mse, n_cells"
        )
    x["source"] = "chemCPA"
    return x

nb11_std = standardize_nb11(extract_nb11_overall(nb11_summary, nb11_overall))
nb12_std = standardize_nb12(nb12_overall)
nb13_std = standardize_nb13(nb13_ci)
chemcpa_std = standardize_chemcpa(chemcpa_metrics)

parity = pd.concat([nb11_std, nb12_std, nb13_std, chemcpa_std], ignore_index=True, sort=False)
parity = parity.sort_values(["split","method"]).reset_index(drop=True)

display(parity)

parity.to_csv(os.path.join(cfg.results_dir, "benchmark_parity_table_nb14.csv"), index=False)

# quick strategic comparison
summary_lines = []
for split in ["test","ood"]:
    sub = parity[parity["split"] == split].copy()
    summary_lines.append(f"[{split.upper()}]")
    for m in ["nb11_residual","nb12_ot","nb13_ot_mean","ctrl_baseline","chemcpa"]:
        row = sub[sub["method"] == m]
        if row.empty:
            continue
        v = row.iloc[0].get("r2_top50", np.nan)
        summary_lines.append(f"  {m:16s} r2_top50={v:.4f}" if pd.notna(v) else f"  {m:16s} r2_top50=NA")
print("\n".join(summary_lines))

if chemcpa_std.empty:
    print("\nBenchmark parity verdict: INCOMPLETE (chemCPA missing).")
else:
    print("\nBenchmark parity verdict: COMPLETE (internal methods + chemCPA present).")

,split,r2_full,mse,r2_top20,r2_top50,method,source,pearson_row,n_cells
0,ood,0.608316,0.363387,0.467559,0.550014,ctrl_baseline,NB12,0.767965,49866.0
1,ood,NaN,NaN,0.467600,0.550000,ctrl_baseline,NB13,0.768000,NaN
2,ood,0.592006,0.378519,0.449129,0.532025,nb11_residual,NB11,NaN,NaN
3,ood,0.617995,0.354408,0.478590,0.559926,nb12_ot,NB12,0.774035,49866.0
4,ood,NaN,NaN,0.478700,0.560000,nb13_ot_mean,NB13,0.774100,NaN
5,test,0.578680,0.398668,0.447537,0.529141,ctrl_baseline,NB12,0.750658,51243.0
6,test,NaN,NaN,0.447500,0.529100,ctrl_baseline,NB13,0.750700,NaN
7,test,0.572822,0.404211,0.438352,0.520113,nb11_residual,NB11,NaN,NaN
8,test,0.626920,0.353021,0.483518,0.562783,nb12_ot,NB12,0.779067,51243.0
9,test,NaN,NaN,0.483500,0.562800,nb13_ot_mean,NB13,0.779000,NaN


[TEST]
  nb11_residual    r2_top50=0.5201
  nb12_ot          r2_top50=0.5628
  nb13_ot_mean     r2_top50=0.5628
  ctrl_baseline    r2_top50=0.5291
[OOD]
  nb11_residual    r2_top50=0.5320
  nb12_ot          r2_top50=0.5599
  nb13_ot_mean     r2_top50=0.5600
  ctrl_baseline    r2_top50=0.5500

Benchmark parity verdict: INCOMPLETE (chemCPA missing).


## Failure decomposition that current artifacts support

This section does not hallucinate unavailable evidence.

It reports:
- **per-cell performance**
- **per-supergroup performance**
- **dose-region composition**
- **drug-family composition** if a reasonable family-like column exists

If detailed per-method prediction artifacts are missing, the notebook will say so instead of faking a result.

In [11]:
# ── Per-cell comparison: NB11 vs NB12 vs NB13 ───────────────────────────
rows = []

# NB11 per-cell
if nb11_per_cell is not None and not nb11_per_cell.empty:
    temp = nb11_per_cell.copy()
    temp.columns = [str(c) for c in temp.columns]
    # try to normalize expected schema
    split_col = "split" if "split" in temp.columns else None
    ct_col = cfg.cell_col if cfg.cell_col in temp.columns else ("cell_type" if "cell_type" in temp.columns else None)
    if split_col and ct_col and "r2_top50" in temp.columns:
        z = temp[[split_col, ct_col, "r2_top50"]].copy()
        z.columns = ["split","cell_type","r2_top50"]
        z["method"] = "nb11_residual"
        rows.append(z)

# NB12 per-cell
if nb12_per_cell is not None and not nb12_per_cell.empty:
    temp = nb12_per_cell.copy()
    temp["method"] = temp["method"].replace({"ot_pilot":"nb12_ot", "ctrl_baseline":"ctrl_baseline"})
    rows.append(temp[["split","cell_type","method","r2_top50"]])

# NB13 per-cell CI
if nb13_ci_per_cell is not None and not nb13_ci_per_cell.empty:
    temp = nb13_ci_per_cell.copy()
    temp["method"] = temp["method"].replace({"ot":"nb13_ot_mean", "ctrl":"ctrl_baseline"})
    temp = temp.rename(columns={"r2_top50_mean":"r2_top50"})
    rows.append(temp[["split","cell_type","method","r2_top50"]])

if rows:
    per_cell_compare = pd.concat(rows, ignore_index=True, sort=False)
    display(per_cell_compare.sort_values(["split","cell_type","method"]))
    per_cell_compare.to_csv(os.path.join(cfg.results_dir, "per_cell_compare_nb14.csv"), index=False)
else:
    per_cell_compare = pd.DataFrame()
    print("Per-cell compare unavailable: required artifacts not found.")

# identify weak cell types for current best internal method
if not per_cell_compare.empty:
    internal_methods = per_cell_compare[per_cell_compare["method"].isin(["nb11_residual","nb12_ot","nb13_ot_mean"])]
    best_internal = (
        internal_methods.sort_values(["split","cell_type","r2_top50"], ascending=[True, True, False])
        .groupby(["split","cell_type"], as_index=False).first()
    )
    print("\nBest internal method by split × cell type:")
    display(best_internal)

,split,cell_type,r2_top50,method
13,ood,A549,0.695625,ctrl_baseline
21,ood,A549,0.695600,ctrl_baseline
3,ood,A549,0.678739,nb11_residual
12,ood,A549,0.703901,nb12_ot
20,ood,A549,0.703900,nb13_ot_mean
15,ood,K562,0.569718,ctrl_baseline
25,ood,K562,0.569700,ctrl_baseline
4,ood,K562,0.560884,nb11_residual
14,ood,K562,0.577491,nb12_ot
24,ood,K562,0.577200,nb13_ot_mean



Best internal method by split × cell type:


,split,cell_type,r2_top50,method
0,ood,A549,0.703901,nb12_ot
1,ood,K562,0.577491,nb12_ot
2,ood,MCF7,0.481900,nb13_ot_mean
3,test,A549,0.707700,nb13_ot_mean
4,test,K562,0.584228,nb12_ot
5,test,MCF7,0.482700,nb13_ot_mean


In [12]:
# ── Per-supergroup comparison from NB12/NB13 ────────────────────────────
sg_compare = None
if nb12_per_sg is not None and not nb12_per_sg.empty:
    temp = nb12_per_sg.copy()
    temp["method"] = temp["method"].replace({"ot_pilot":"nb12_ot", "ctrl_baseline":"ctrl_baseline"})
    sg_compare = temp.copy()

if sg_compare is not None:
    display(sg_compare.sort_values(["split","supergroup","method"]))
    sg_compare.to_csv(os.path.join(cfg.results_dir, "per_supergroup_compare_nb14.csv"), index=False)

if nb13_supergroup is not None and not nb13_supergroup.empty:
    print("\nNB13 OOD supergroup gain table:")
    display(nb13_supergroup.sort_values("gain_r2_top50", ascending=False))
    nb13_supergroup.to_csv(os.path.join(cfg.results_dir, "ood_supergroup_gain_nb14.csv"), index=False)

# OT-favorable supergroups from NB13
if nb13_supergroup is not None and not nb13_supergroup.empty:
    ot_favorable_sg = nb13_supergroup.loc[
        nb13_supergroup["gain_r2_top50"] >= cfg.ot_gain_threshold_group,
        "supergroup"
    ].tolist()
    print(f"OT-favorable OOD supergroups (gain >= {cfg.ot_gain_threshold_group:.2f}): {ot_favorable_sg}")
else:
    ot_favorable_sg = []

,label,n_cells,r2_full,pearson_row,mse,r2_top20,r2_top50,split,supergroup,n_drugs,method
9,ctrl_baseline_sg0_ood,25265,0.624345,0.778014,0.346728,0.482203,0.566263,ood,0,13,ctrl_baseline
8,ot_pilot_sg0_ood,25265,0.624816,0.778271,0.346293,0.483765,0.567744,ood,0,13,nb12_ot
11,ctrl_baseline_sg2_ood,5911,0.607491,0.767707,0.346206,0.457159,0.546091,ood,2,3,ctrl_baseline
10,ot_pilot_sg2_ood,5911,0.608875,0.768591,0.344985,0.460597,0.547618,ood,2,3,nb12_ot
13,ctrl_baseline_sg3_ood,1696,0.258474,0.559419,0.710823,0.207868,0.255903,ood,3,1,ctrl_baseline
12,ot_pilot_sg3_ood,1696,0.473308,0.690823,0.504885,0.398285,0.433190,ood,3,1,nb12_ot
15,ctrl_baseline_sg5_ood,14399,0.635993,0.784985,0.348188,0.492346,0.576189,ood,5,8,ctrl_baseline
14,ot_pilot_sg5_ood,14399,0.636184,0.785048,0.348005,0.492627,0.576420,ood,5,8,nb12_ot
17,ctrl_baseline_sg7_ood,2595,0.529450,0.712581,0.421985,0.380866,0.447737,ood,7,2,ctrl_baseline
16,ot_pilot_sg7_ood,2595,0.562815,0.738458,0.392064,0.443780,0.503145,ood,7,2,nb12_ot



NB13 OOD supergroup gain table:


,supergroup,n_ood_drugs,n_ood_cells,ot_r2_top50,ctrl_r2_top50,gain_r2_top50,mean_routing_dist
0,3,1,1696,0.4332,0.2559,0.1773,0.1873
1,7,2,2595,0.5031,0.4477,0.0554,0.5724
2,0,13,25265,0.5677,0.5663,0.0015,0.2368
3,2,3,5911,0.5476,0.5461,0.0015,0.3743
4,5,8,14399,0.5764,0.5762,0.0002,0.2549


OT-favorable OOD supergroups (gain >= 0.03): [3, 7]


In [13]:
# ── Dose-region and family composition (not performance unless available) ─
obs2 = obs.copy()

# dose bins: fixed quantiles on all cells for audit
dose_vals = pd.to_numeric(obs2[cfg.dose_col], errors="coerce")
obs2["dose_bin"] = pd.qcut(dose_vals.rank(method="first"), q=5, labels=[f"Q{i}" for i in range(1,6)])

dose_comp = (
    obs2.groupby([cfg.split_col, "dose_bin"]).size().reset_index(name="n_cells")
    .pivot(index="dose_bin", columns=cfg.split_col, values="n_cells")
    .fillna(0).astype(int)
)
display(dose_comp)
dose_comp.to_csv(os.path.join(cfg.results_dir, "dose_composition_nb14.csv"))

if family_col is not None:
    fam_comp = (
        obs2.groupby([cfg.split_col, family_col]).size().reset_index(name="n_cells")
        .sort_values("n_cells", ascending=False)
    )
    fam_top = fam_comp.groupby(cfg.split_col).head(15)
    print(f"Using family-like column: {family_col}")
    display(fam_top)
    fam_top.to_csv(os.path.join(cfg.results_dir, "family_composition_top_nb14.csv"), index=False)
else:
    print("No family-like column found. Skipping family composition table.")

split_scaffold,control,ood,test,train
dose_bin,,,,
Q1,13004,8401,8523,41000
Q2,0,10021,10608,50299
Q3,0,10233,10708,49987
Q4,0,10324,10639,49965
Q5,0,10887,10765,49276


Using family-like column: pathway_level_1


,split_scaffold,pathway_level_1,n_cells
55,train,Epigenetic regulation,59253
66,train,Tyrosine kinase signaling,35621
58,train,JAK/STAT signaling,30064
54,train,DNA damage & DNA repair,22672
53,train,Cell cycle regulation,21691
21,ood,Epigenetic regulation,18724
16,control,Vehicle,13004
61,train,Nuclear receptor signaling,12145
38,test,Epigenetic regulation,11509
64,train,Protein folding & Protein degradation,9387


## Benchmark-beating readiness and hybrid-method blueprint

This section turns the current evidence into a **non-hallucinated recommendation**.

It will answer:
- Is the dataset broken?  
- Is internal progress real?  
- Is parity complete?  
- What should the next method look like?

In [14]:
# ── Internal progress check ─────────────────────────────────────────────
readiness = []

# dataset okay?
dataset_ok = True
readiness.append(("dataset_ok_for_rigorous_scaffold_study", dataset_ok,
                  "Large enough and diverse enough for serious scaffold/OOD evaluation; not enough alone for universal SOTA claim."))

# internal baseline survives?
nb11_test = parity.query("method == 'nb11_residual' and split == 'test'")["r2_top50"]
nb11_ood  = parity.query("method == 'nb11_residual' and split == 'ood'")["r2_top50"]
nb11_survives = (not nb11_test.empty and not nb11_ood.empty and nb11_test.iloc[0] > 0 and nb11_ood.iloc[0] > 0)
readiness.append(("nb11_scaffold_survives", bool(nb11_survives),
                  "Residual baseline remains positive on scaffold test/OOD."))

# OT stronger than internal residual at overall summary level?
ot_test = parity.query("method == 'nb13_ot_mean' and split == 'test'")["r2_top50"]
ot_ood  = parity.query("method == 'nb13_ot_mean' and split == 'ood'")["r2_top50"]
ot_beats_nb11 = (
    not ot_test.empty and not ot_ood.empty and
    not nb11_test.empty and not nb11_ood.empty and
    (ot_test.iloc[0] > nb11_test.iloc[0]) and
    (ot_ood.iloc[0] > nb11_ood.iloc[0])
)
readiness.append(("ot_beats_nb11_overall_metric_summary", bool(ot_beats_nb11),
                  "Cross-notebook overall metric comparison only; useful strategically, not a paired significance test."))

# parity complete?
parity_complete = not chemcpa_std.empty
readiness.append(("benchmark_parity_complete", parity_complete,
                  "chemCPA same-benchmark comparison present." if parity_complete else "chemCPA parity missing; cannot claim benchmark-beating."))

# pathway status
pathway_ok = False
if nb13_pathway is not None and not nb13_pathway.empty:
    # require at least one non-zero / non-null recall-like field
    candidate_cols = [c for c in nb13_pathway.columns if "pathway_recall" in c or "jaccard" in c]
    if candidate_cols:
        vals = pd.to_numeric(nb13_pathway[candidate_cols].stack(), errors="coerce")
        pathway_ok = vals.notna().any()
readiness.append(("pathway_validation_present", pathway_ok,
                  "Pathway/DEG evidence found." if pathway_ok else "Pathway validation missing, broken, or incomplete."))

readiness_df = pd.DataFrame(readiness, columns=["check","status","note"])
display(readiness_df)
readiness_df.to_csv(os.path.join(cfg.results_dir, "benchmark_readiness_nb14.csv"), index=False)

# ── Hybrid blueprint ────────────────────────────────────────────────────
blueprint_rows = []

overall_test_gain = float(ot_test.iloc[0] - nb11_test.iloc[0]) if (not ot_test.empty and not nb11_test.empty) else np.nan
overall_ood_gain  = float(ot_ood.iloc[0]  - nb11_ood.iloc[0])  if (not ot_ood.empty and not nb11_ood.empty) else np.nan

blueprint_rows.append({
    "component": "residual_backbone",
    "decision": "KEEP",
    "reason": "NB11 scaffold baseline survives; residual formulation remains the stable anchor."
})
blueprint_rows.append({
    "component": "pure_ot_as_whole_solution",
    "decision": "DO_NOT_TRUST_AS_FINAL",
    "reason": "OT shows real gains, but gains are heterogeneous and pathway/benchmark parity remain incomplete."
})
blueprint_rows.append({
    "component": "selective_ot_correction",
    "decision": "PROMOTE",
    "reason": f"OT-favorable regions exist (overall Δtest={overall_test_gain:+.4f}, Δood={overall_ood_gain:+.4f}; supergroup gains are uneven)."
})
blueprint_rows.append({
    "component": "stronger_molecular_representation",
    "decision": "ADD",
    "reason": "Current bottleneck appears more like information/method mismatch than training instability."
})
blueprint_rows.append({
    "component": "next_method_shape",
    "decision": "HYBRID",
    "reason": "Residual backbone + stronger drug representation + OT correction only where OT is demonstrably helpful."
})

blueprint_df = pd.DataFrame(blueprint_rows)
display(blueprint_df)
blueprint_df.to_csv(os.path.join(cfg.results_dir, "hybrid_blueprint_nb14.csv"), index=False)

,check,status,note
0,dataset_ok_for_rigorous_scaffold_study,True,Large enough and diverse enough for serious scaffold/OOD evaluation; not enough alone for universal SOTA claim.
1,nb11_scaffold_survives,True,Residual baseline remains positive on scaffold test/OOD.
2,ot_beats_nb11_overall_metric_summary,True,"Cross-notebook overall metric comparison only; useful strategically, not a paired significance test."
3,benchmark_parity_complete,False,chemCPA parity missing; cannot claim benchmark-beating.
4,pathway_validation_present,True,Pathway/DEG evidence found.


,component,decision,reason
0,residual_backbone,KEEP,NB11 scaffold baseline survives; residual formulation remains the stable anchor.
1,pure_ot_as_whole_solution,DO_NOT_TRUST_AS_FINAL,"OT shows real gains, but gains are heterogeneous and pathway/benchmark parity remain incomplete."
2,selective_ot_correction,PROMOTE,"OT-favorable regions exist (overall Δtest=+0.0427, Δood=+0.0280; supergroup gains are uneven)."
3,stronger_molecular_representation,ADD,Current bottleneck appears more like information/method mismatch than training instability.
4,next_method_shape,HYBRID,Residual backbone + stronger drug representation + OT correction only where OT is demonstrably helpful.


In [15]:
# ── Final blunt summary ─────────────────────────────────────────────────
print("=" * 78)
print("NB14 FINAL VERDICT")
print("=" * 78)

print("\n1) Dataset status")
print("   - Current dataset is OK for a rigorous scaffold/OOD benchmark study.")
print("   - It is NOT enough by itself for a universal SOTA claim across all perturbation settings.")

print("\n2) Internal method status")
if nb11_survives:
    print("   - NB11 residual baseline survives scaffold split.")
else:
    print("   - NB11 residual baseline does NOT look stable; re-check artifacts immediately.")

if ot_beats_nb11:
    print("   - NB13 OT mean is higher than NB11 residual on overall test and OOD summary metrics.")
    print("   - Treat this as strong strategic evidence, not final paired statistical proof.")
else:
    print("   - OT has not clearly exceeded NB11 residual at the overall summary level.")

print("\n3) Benchmark parity status")
if parity_complete:
    print("   - chemCPA parity is present. You can now honestly ask whether you are winning the benchmark.")
else:
    print("   - chemCPA parity is MISSING. You cannot honestly claim benchmark-beating yet.")

print("\n4) Recommended next method")
print("   - Do NOT make another pure OT notebook.")
print("   - Next serious candidate should be HYBRID:")
print("       residual backbone + stronger molecular representation + selective OT correction")

print("\n5) Immediate next step")
if parity_complete:
    print("   - Use the parity table and failure decomposition to design the first benchmark-beating hybrid attempt.")
else:
    print("   - First run chemCPA on the same scaffold split and save a compatible metrics CSV.")
    print("   - Then rerun NB14 and only after that design the benchmark-beating hybrid attempt.")

print("\nSaved outputs:")
for fn in [
    "artifact_audit_nb14.csv",
    "dataset_summary_nb14.csv",
    "benchmark_parity_table_nb14.csv",
    "per_cell_compare_nb14.csv",
    "per_supergroup_compare_nb14.csv",
    "ood_supergroup_gain_nb14.csv",
    "benchmark_readiness_nb14.csv",
    "hybrid_blueprint_nb14.csv",
]:
    path = os.path.join(cfg.results_dir, fn)
    if os.path.exists(path):
        print("  -", path)

NB14 FINAL VERDICT

1) Dataset status
   - Current dataset is OK for a rigorous scaffold/OOD benchmark study.
   - It is NOT enough by itself for a universal SOTA claim across all perturbation settings.

2) Internal method status
   - NB11 residual baseline survives scaffold split.
   - NB13 OT mean is higher than NB11 residual on overall test and OOD summary metrics.
   - Treat this as strong strategic evidence, not final paired statistical proof.

3) Benchmark parity status
   - chemCPA parity is MISSING. You cannot honestly claim benchmark-beating yet.

4) Recommended next method
   - Do NOT make another pure OT notebook.
   - Next serious candidate should be HYBRID:
       residual backbone + stronger molecular representation + selective OT correction

5) Immediate next step
   - First run chemCPA on the same scaffold split and save a compatible metrics CSV.
   - Then rerun NB14 and only after that design the benchmark-beating hybrid attempt.

Saved outputs:
  - /content/drive/MyDr

## Optional: chemCPA metrics file format

If `chemCPA` has already been run elsewhere, save a CSV at:

`/content/drive/MyDrive/ChemDFM/results_chemcpa_scaffold/chemcpa_scaffold_metrics.csv`

with at least these columns:

- `method`
- `split`
- `r2_top50`

Recommended full schema:

- `method`
- `split`
- `r2_top50`
- `r2_top20`
- `pearson_row`
- `r2_full`
- `mse`
- `n_cells`

Example rows:

| method  | split | r2_top50 | r2_top20 | pearson_row |
|---------|-------|----------|----------|-------------|
| chemcpa | test  | 0.0000   | 0.0000   | 0.0000      |
| chemcpa | ood   | 0.0000   | 0.0000   | 0.0000      |

Then rerun NB14.